In [0]:
from pathlib import Path

github_root = Path(
    "/Workspace/Users/prxaura@gmail.com/materials-platform/lammps"
)

runs_dir = github_root / "Si" / "runs"

print(runs_dir)
print(runs_dir.exists())

In [0]:
import json

records = []

for run_dir in runs_dir.iterdir():

    if not run_dir.is_dir():
        continue

    run_file = run_dir / "run.json"

    if not run_file.exists():
        continue

    with open(run_file) as f:
        data = json.load(f)
    print(data)
    data["run_id"] = run_dir.name
    records.append(data)

df = spark.createDataFrame(records)

display(df)

In [0]:
from pathlib import Path

from pyspark.sql import functions as F
# Collect all lattice-parameter log files
lp_log_files = []

for run_dir in runs_dir.iterdir():

    if not run_dir.is_dir():
        continue

    lp_dir = run_dir / "results" / "lattice_parameter"

    if not lp_dir.exists():
        continue

    files = list(lp_dir.rglob("*.log"))

    lp_log_files.extend(files)


print(f"Found {len(lp_log_files)} log files")


# Read all log files into ONE Spark DataFrame
raw_dfs = []

for f in lp_log_files:

    df = (
        spark.read
        .text(str(f))
        .withColumn("source_file", F.lit(str(f)))
    )

    raw_dfs.append(df)


raw_df = raw_dfs[0]

for df in raw_dfs[1:]:
    raw_df = raw_df.unionByName(df)

display(raw_df)

In [0]:
thermo_df = (
    raw_df
    .filter(
        F.col("value").rlike(
            r"^\s*\d+\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s*$"
        )
    )
    .select(
        "value",
        "source_file"
    )
)

display(thermo_df)

In [0]:

thermo_df = (
    raw_df
    .filter(
        F.col("value").rlike(
            r"^\s*\d+\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+"
            r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s*$"
        )
    )
)

display(thermo_df)

In [0]:
thermo_df = (
    thermo_df
    .select(
        F.split(F.trim(F.col("value")), r"\s+").alias("cols"),
        "source_file"
    )
    .select(
        F.col("cols")[0].cast("long").alias("step"),
        F.col("cols")[1].cast("double").alias("temperature"),
        F.col("cols")[2].cast("double").alias("poteng"),
        F.col("cols")[3].cast("double").alias("pressure"),
        F.col("cols")[4].cast("double").alias("volume"),
        "source_file"
    )
)

display(thermo_df)

In [0]:
thermo_df = (
    thermo_df
    .withColumn(
        "run_id",
        F.regexp_extract(
            F.col("source_file"),
            r"runs/([0-9a-fA-F-]{36})/",
            1
        )
    )
    .withColumn("element", F.lit("Si"))
)

display(thermo_df)

In [0]:
%sql
create table if not exists workspace.B_silicon

In [0]:
thermo_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "B_silicon.lattice_parameter_simulation_runs"
    )

In [0]:
%sql

Select distinct run_id from workspace.B_silicon.lattice_parameter_simulation_runs